# Example 2.4.2 / 1: JSON to Dictionaries

Companion notebook for **2.4.2 Data from JSON and Other Files** in the AMPL lab Documentation, section "JSON to Dictionaries". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## JSON to Dictionaries

The example starts with JSON text so it is self-contained; a real application commonly uses
`json.load(file)` for a file or `response.json()` for an API response.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param market {PROD} >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
import json

raw = json.loads('''{
  "products": [
    {"name": "bands", "profit": 25, "market": 6000},
    {"name": "coils", "profit": 30, "market": 4000}
  ]
}''')

records = raw["products"]
names = [row["name"] for row in records]
if len(names) != len(set(names)):
    raise ValueError("duplicate product name")

ampl.set["PROD"] = names
ampl.param["profit"] = {row["name"]: row["profit"] for row in records}
ampl.param["market"] = {row["name"]: row["market"] for row in records}

In [5]:
# run.py
print(ampl.param["profit"].to_dict())

{'bands': 25, 'coils': 30}


### Pure AMPL: data.dat and model.run

The same validated instance in AMPL data syntax:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param profit := bands 25  coils 30;
param market := bands 6000  coils 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display profit;

profit [*] :=
bands  25
coils  30
;

